In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('livros.csv')

In [ ]:
traducao = {
    'Book':'livro',
    'Author(s)':'autor',
    'Original language':'idioma_original',
    'First published':'ano_publicacao',
    'Approximate sales in millions':'vendas',
    'Genre':'genero'
}

In [ ]:
df.rename(columns=traducao, inplace=True)

In [ ]:
autores_unicos = pd.DataFrame(df['autor'].unique(), columns=['autor'])

In [ ]:
autores_unicos['autor_id'] = autores_unicos.index + 1

In [ ]:
df = df.merge(autores_unicos, on = 'autor', how = 'left')

In [ ]:
generos_unicos = pd.DataFrame(df['genero'].unique(), columns=['genero'])

In [ ]:
generos_unicos['genero_id'] = generos_unicos.index + 1

In [ ]:
df = df.merge(generos_unicos, on = 'genero', how = 'left')

In [ ]:
df['genero'] = df['genero'].fillna('Unknown')

In [ ]:
generos = pd.DataFrame(df['genero'].unique(), columns=['genero'])

In [ ]:
!pip install -q deep-translator

from deep_translator import GoogleTranslator

# Supondo que você tenha um DataFrame chamado generos com a coluna 'genero'
generos['genero_pt'] = generos['genero'].apply(
    lambda x: GoogleTranslator(source='auto', target='pt').translate(x)
)

In [ ]:
generos['genero_pt'] = generos['genero_pt'].replace('Novella', 'Novela')

generos['genero_pt'] = generos['genero_pt'].str.lower()

In [ ]:
autores = pd.DataFrame(df['autor'].unique(), columns=['nome'])

In [ ]:
api = 'https://raw.githubusercontent.com/guilhermeonrails/datas-csv/refs/heads/main/comentarios.json'
df_comentarios = pd.read_json(api)

df_comentarios = df_comentarios.merge(
    df[['livro']].reset_index().rename(columns={'index': 'id_livro'}),
    on='livro',
    how='left'
)
df_comentarios['id_livro'] += 1

def format_value(value):
    if pd.isna(value):
        return 'NULL'
    elif isinstance(value, str):
        value = value.replace("'", "''")  # Escapa aspas simples para SQL
        return f"'{value}'"
    else:
        return str(value)

output_file = "comentarios.sql"

In [ ]:
with open('autores.sql', 'w', encoding='utf-8') as f:
  for _, row in autores.iterrows():
        nome = row['nome'].replace("'", "''")  # aspas simples para SQL
        f.write(f"INSERT INTO autores (nome) VALUES ('{nome}');\n")

In [ ]:
with open('generos.sql', 'w', encoding='utf-8') as f:
    for nome in generos['genero_pt']:
        nome_escapado = nome.replace("'", "''")  # aspas simples para SQL
        f.write(f"INSERT INTO generos (nome) VALUES ('{nome_escapado}');\n")

In [ ]:
with open('livros.sql', 'w', encoding='utf-8') as f:
    for _, row in df.iterrows():
        nome_livro = row['livro'].replace("'", "''")  # Escapa aspas simples
        idioma = row['idioma_original'].replace("'", "''")
        ano = int(row['ano_publicacao'])
        vendas = float(row['vendas'])
        autor_id = int(row['autor_id'])
        genero_id = int(row['genero_id'])

        sql = (
            f"INSERT INTO livros (nome, idioma, ano_publicacao, vendas, autor_id, genero_id) "
            f"VALUES ('{nome_livro}', '{idioma}', {ano}, {vendas:.2f}, {autor_id}, {genero_id});\n"
        )
        f.write(sql)

In [ ]:
with open(output_file, 'w', encoding='utf-8') as f:
    for _, row in df_comentarios.iterrows():
        values = (
            format_value(row['id_livro']),
            format_value(row['nome']),
            format_value(row['sobrenome']),
            format_value(row['comentario'])
        )
        sql = f"INSERT INTO comentarios (livro_id, nome, sobrenome, comentario) VALUES ({', '.join(values)});\n"
        f.write(sql)

print(f"Arquivo '{output_file}' gerado com sucesso.")


Arquivo 'comentarios.sql' gerado com sucesso.
